# Enrichment analysis

Reference:
- [module documentation](https://analytics-core.readthedocs.io/stable/reference/acore.enrichment_analysis.html)
- [api-example for enrichment analysis](https://analytics-core.readthedocs.io/stable/api_examples/enrichment_analysis.html)

In [ ]:
import acore
import pandas as pd

## List relevant files
- ms2query based annotations (contains smiles and inchikeys)
- ancova results (contains feature IDs and p-values)

In [ ]:
fname_ms2query = "results_prepared/output_ms2query_Linked_data.tsv"
fname_ancova = "results_prepared/ancova_results.csv"
fname_pathways_map = "results_prepared/pathways_map.tsv"
fname_inchikey_to_kegg = "results_prepared/inchikey_to_kegg.csv"
fname_annotations = "results_prepared/link_compound_pathway.tsv"

## Kegg annotations
Can be downloaded from KEGG:
- https://rest.kegg.jp/link/compound/pathway

In [ ]:
annotations = pd.read_csv(
    fname_annotations,
    sep="\t",
    header=None,
    names=[
        "pathway_id",
        "compound_id",
    ],
)
_ = annotations.insert(1, "Source", "KEGG")
annotations["pathway_id"] = (
    annotations["pathway_id"].str.strip().str.replace("path:", "")
)
annotations["compound_id"] = (
    annotations["compound_id"].str.strip().str.replace("cpd:", "")
)
annotations.set_index("compound_id", inplace=True)
annotations

## Pathway mapping: fetch names

In [ ]:
pathways_map = pd.read_csv(
    fname_pathways_map, sep="\t", header=None, names=["pathway_id", "pathway_name"]
)
pathways_map.head()

exclude some generic pathways?

In [ ]:
mask = pathways_map["pathway_name"].str.contains("pathways", case=False)
pathways_map.loc[mask]

Can be downloaded from KEGG:
- https://rest.kegg.jp/link/compound/pathway

## Filtering pathways
filter some generic pathways if you want.

In [ ]:
view = annotations.groupby("pathway_id").size().sort_values(ascending=False)
view.plot(kind="line", figsize=(10, 5), marker=".")
view

Some pathway maps:
- [map01100](https://rest.kegg.jp/get/path:map01100/image)
- [map01110](https://rest.kegg.jp/get/path:map01110/image)
- [map01120](https://rest.kegg.jp/get/path:map01120/image)
- [map01130](https://rest.kegg.jp/get/path:map01130/image)

For example map00010:
- see conf map [conf map of map00010](https://rest.kegg.jp/get/path:map00010/conf)
![path:map00010/image](https://rest.kegg.jp/get/path:map00010/image)

Additional information for map00010 and map00030:
- https://rest.kegg.jp/get/path:map00030+path:map00010

In [ ]:
ms2query_results = pd.read_csv(fname_ms2query, index_col=0, sep="\t").drop_duplicates(
    subset=["inchikey", "smiles"]
)
ms2query_results.head()

In [ ]:
inchikey_to_kegg = pd.read_csv(fname_inchikey_to_kegg, index_col=0).astype({"id": str})
inchikey_to_kegg

## Reload analysis of covariance (ANCOVA) results


In [ ]:
ancova = pd.read_csv(fname_ancova, index_col=0)
ancova.index = ancova.index.astype(str)
ancova

Let's see if we could identify features from the differential regulations
analysis using the available MS2 annotations. We will use the `inchikey_to_kegg`
mapping from `3_enrichment_analysis_fetch_kegg.ipynb`, which was pre-executed and the
results stored. Rerun with new data!

In [ ]:
inchikey_to_kegg  # .loc[ids_found_inMS2]

In [ ]:
regex_filter = "pval|padj|reject|FC"
ids_found_inMS2 = inchikey_to_kegg["id"].unique().tolist()
ids_found_inMS_also_in_ancova = list(set(ids_found_inMS2).intersection(ancova.index))
ancova.loc[ids_found_inMS_also_in_ancova].filter(regex=regex_filter).sort_values(
    "pvalue"
)

Make the few identified features significant for illustration purposes.

In [ ]:
ancova.loc[ids_found_inMS_also_in_ancova, "pvalue"] = 0.01
ancova.loc[ids_found_inMS_also_in_ancova].filter(regex=regex_filter).sort_values(
    "pvalue"
)

Let's manually update some compound IDs for the few features we identified.
- choose one compound per feature

In [ ]:
inchikey_to_kegg_of_interest = inchikey_to_kegg.loc[inchikey_to_kegg["id"].isin(ids_found_inMS_also_in_ancova)]
inchikey_to_kegg_of_interest

In [ ]:
rename_index = {
    "4051789042754256385": "C12048",
    "7939233295536706460": "C10358",
    "356441345885270616": "C03194",  # C02962
}
ancova = ancova.rename(index=rename_index)

In [ ]:
ancova.loc[rename_index.values()].filter(regex=regex_filter).sort_values("pvalue")

## Enrichment analysis
We will use the annotations fetched from KEGG to perform the enrichment analysis.

In [ ]:
annotations = annotations.rename_axis("identifier").reset_index()
annotations

In [ ]:
ret = acore.enrichment_analysis.run_up_down_regulation_enrichment(
    regulation_data=ancova.rename_axis("identifier").reset_index(),
    annotation=annotations,
    identifier="identifier",
    annotation_col="pathway_id",
    pval_col="pvalue",
    min_detected_in_set=1,
    lfc_cutoff=0.0001,
)
ret

In [ ]:
ancova.loc[rename_index.values()].filter(regex=regex_filter).sort_values("pvalue")

Why do we only see one compound?

In [ ]:
annotations.loc[annotations.identifier.isin(inchikey_to_kegg_of_interest.kegg_id)]